# Fase 2 · Obtención, limpieza y transformación de los datos

**Proyecto:** Fallas recurrentes de transmisión en la red de un operador móvil (NOC de WOM, octubre de 2026)
**Curso:** MCDI500 · Programación para la Ciencia de Datos · Magíster en Ciencia de Datos e IA, UNAB
**Integrante:** Edgar Arcia Velásquez (Grupo 6, trabajo individual) · **Docente:** Dr. Omar Salinas Silva

---

Este notebook toma el archivo de `data/raw` y entrega en `data/processed` tres productos:

1. `incidencias_limpio.csv`: las 21.542 incidencias limpias y legibles, con las variables derivadas.
2. `incidencias_modelo.csv`: la misma tabla en formato numérico (ordinales, one-hot y escaladas), para las Fases 3 y 4.
3. `recurrencia_tx_por_elemento.csv`: **la respuesta a la pregunta del proyecto**. Una fila por elemento de red con sus fallas de transmisión con pérdida de servicio y los minutos que estuvo fuera de servicio.

El flujo es **obtener → explorar → limpiar → transformar → escalar → validar → guardar**. La regla en
todo el notebook: **primero medir, después decidir, y dejar la decisión registrada con su cifra**.

## Introducción

La Fase 1 dejó definida la pregunta y midió, sin corregir, los problemas del archivo. Esta fase los resuelve:

| Problema medido en F1 | Dónde se resuelve |
|---|---|
| 43.811 celdas con tildes rotas y un encabezado con restos de HTML (`</div>`) | 3.1 y 3.2 |
| La misma categoría escrita de varias formas (`NA`, `N/A`, `na`, `no aplica`...) | 3.3 |
| 26 columnas con más de 60 % vacío | 3.4 |
| Duración como texto `h:mm:ss` y 2 duraciones negativas | 3.5 |
| 5.318 duraciones vacías (incidencias aún no normalizadas) | 3.6 |
| 2.708 filas con TT repetido | 2.3 |
| Criticidad e impacto con un orden que el alfabeto no respeta | 4.2 |

### Librerías

| Librería | Uso en este notebook |
|---|---|
| pandas | carga, perfil, limpieza, agrupaciones y tablas |
| NumPy | logaritmos, cuartiles y comparaciones numéricas |
| scikit-learn | `OneHotEncoder` y los tres escaladores, con el patrón `fit` / `transform` |
| matplotlib | gráficos de control que quedan en `docs/figuras/` |

## Configuración del entorno y de las rutas

Misma verificación que en la Fase 1. Si el kernel no es el del proyecto, el notebook se detiene aquí
y no veinte celdas después.

In [ ]:
import sys
import json
import platform
import importlib
import subprocess
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler

SEMILLA = 42
np.random.seed(SEMILLA)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

print(f"Python {platform.python_version()} · entorno virtual: {'sí' if '.venv' in sys.executable else 'NO'}")
for lib in (np, pd, matplotlib, sklearn):
    print(f"  {lib.__name__:<11} {lib.__version__}")
assert ".venv" in sys.executable, "El kernel no es el del proyecto: Kernel > Change Kernel > Python (mcdi500)"

def localizar_raiz(inicio=None):
    '''Primera carpeta hacia arriba que contiene requirements.txt (la raíz del proyecto).'''
    actual = Path(inicio or Path.cwd()).resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / "requirements.txt").is_file():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz del proyecto (requirements.txt)")

RAIZ = localizar_raiz()
DIR_RAW, DIR_PROC, DIR_DOCS = RAIZ / "data" / "raw", RAIZ / "data" / "processed", RAIZ / "docs"
DIR_FIG = DIR_DOCS / "figuras"
for carpeta in (DIR_PROC, DIR_FIG):
    carpeta.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(RAIZ / "src"))
import utilidades
importlib.reload(utilidades)
print("Raíz del proyecto:", RAIZ.name)

### Bitácora de decisiones

Una función registra cada decisión con su motivo y su impacto medido. Al final se escribe en
`docs/bitacora_decisiones.md`, así el informe cita cifras que salen del notebook y no de la memoria.

In [ ]:
BITACORA = []

def registrar(paso, decision, motivo, impacto):
    '''Agrega una decisión a la bitácora y la devuelve para mostrarla.'''
    entrada = {"paso": paso, "decision": decision, "motivo": motivo, "impacto": impacto}
    BITACORA.append(entrada)
    print(f"[bitácora {paso}] {decision} → {impacto}")
    return entrada

## 1. Obtención de los datos

**Qué hace este paso.** Carga el archivo con la misma clase que usó la Fase 1 (`ConjuntoDatos`),
como texto y sin convertir nada, para medir el estado original. Calcula la **huella SHA-256** del
archivo: al final del notebook se vuelve a calcular para demostrar que `data/raw` no se modificó.
También compara la forma con la registrada en `docs/metadatos_fase1.json`.

In [ ]:
meta_f1 = json.loads((DIR_DOCS / "metadatos_fase1.json").read_text(encoding="utf-8"))
conjunto = utilidades.ConjuntoDatos(RAIZ / meta_f1["dataset"]["archivo"])
HUELLA_RAW = utilidades.huella_sha256(conjunto.ruta)

crudo = conjunto.cargar(como_texto=True)
FILAS_ORIGINALES, COLUMNAS_ORIGINALES = crudo.shape
assert (FILAS_ORIGINALES, COLUMNAS_ORIGINALES) == (meta_f1["dataset"]["filas"], meta_f1["dataset"]["columnas"]), \
    "El archivo no tiene la forma que registró la Fase 1"
print(conjunto)
print(f"{FILAS_ORIGINALES:,} filas × {COLUMNAS_ORIGINALES} columnas · igual a la Fase 1".replace(",", "."))
print("SHA-256:", HUELLA_RAW[:16], "...")

## 2. Exploración inicial

### 2.1 Perfil de vacíos con todas sus formas

En F1 se vio que "sin dato" se escribe de varias formas. Aquí se cuentan **todas** como vacío
(`utilidades.es_vacio`), para saber cuánto falta de verdad en cada columna.

In [ ]:
def perfil_vacios(df):
    '''% de celdas sin dato (en cualquiera de sus formas) y cantidad de valores distintos por columna.'''
    vacio = df.apply(lambda col: col.map(utilidades.es_vacio))
    return pd.DataFrame({"pct_vacio": (vacio.mean() * 100).round(1), "n_distintos": df.nunique()}) \
             .sort_values("pct_vacio", ascending=False)

perfil_inicial = perfil_vacios(crudo)
perfil_inicial.index = [utilidades.reparar_texto(c) for c in perfil_inicial.index]
print("Columnas con más de 95 % sin dato:", int((perfil_inicial["pct_vacio"] > 95).sum()))
print("Columnas completas (0 % sin dato):", int((perfil_inicial["pct_vacio"] == 0).sum()))
perfil_inicial.head(30)

In [ ]:
def grafico_vacios(perfil, ruta):
    '''Barras horizontales con el % sin dato de cada columna que tiene algún vacío.'''
    p = perfil[perfil["pct_vacio"] > 0].sort_values("pct_vacio")
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.barh(p.index, p["pct_vacio"], color="#3B6FD8")
    ax.axvline(95, color="#C0392B", linestyle="--", linewidth=1)
    ax.text(95.5, 0, "95 %", color="#C0392B", fontsize=8)
    ax.set_xlabel("% de filas sin dato")
    ax.set_title("Vacíos por columna (archivo original)")
    fig.tight_layout()
    fig.savefig(ruta, dpi=150)
    plt.show()

grafico_vacios(perfil_inicial, DIR_FIG / "f2_vacios.png")

### 2.2 ¿De dónde vienen las duraciones vacías?

La Fase 1 verificó que todas las duraciones vacías son incidencias **Abiertas** o **Pendientes**.
Aquí se mide además en qué tipo de incidencia caen, porque eso dice si afectan o no a la pregunta.

In [ ]:
def tasa_vacio_por_grupo(df, variable, grupo):
    '''% de filas sin dato en `variable` dentro de cada categoría de `grupo`.'''
    return (df[variable].map(utilidades.es_vacio).groupby(df[grupo]).mean() * 100).round(1).rename(f"% sin {variable}")

col_dur_crudo = [c for c in crudo.columns if utilidades.reparar_texto(c) == "Tiempo de duración de la incidencia"][0]
print(tasa_vacio_por_grupo(crudo, col_dur_crudo, "Estado").to_string(), "\n")
es_tx_crudo = crudo["Tipo Incidencia NOC"].isin(["Tx Propia", "Tx Proveedor"])
vacias = crudo[col_dur_crudo].map(utilidades.es_vacio)
print(f"Duraciones vacías en fallas de transmisión: {int((vacias & es_tx_crudo).sum())} de {int(vacias.sum()):,}".replace(",", "."))
print("Tipo de incidencia de las vacías:", crudo.loc[vacias, "Tipo Incidencia NOC"].map(utilidades.reparar_texto).value_counts().head(3).to_dict())

**Lectura.** Casi todas las duraciones vacías son *Incidencias Automáticas* en estado *Pendiente
Automático*: alarmas que el sistema abrió y que nadie ha cerrado. Solo unas pocas son de
transmisión. El vacío **depende del estado** (no ocurre al azar), así que rellenarlo con un valor
central inventaría cierres que no existen. Se trata en 3.6.

### 2.3 Duplicados y TT repetidos

F1 encontró 2.708 filas cuyo TT aparece en otra fila. Antes de borrar nada hay que ver si son
duplicados de verdad o incidencias distintas que comparten un ticket.

In [ ]:
CLAVE = ["TT", "Network Element", "Fecha de Inicio de incidencia"]
crudo_tt = crudo.rename(columns={crudo.columns[0]: "TT"})
repetidos = crudo_tt[crudo_tt["TT"].duplicated(keep=False)]
print("Filas duplicadas exactas      :", int(crudo_tt.duplicated().sum()))
print("TT que aparecen más de una vez:", repetidos["TT"].nunique(), "en", len(repetidos), "filas")
print("Filas repetidas en la clave TT + elemento + inicio:", int(crudo_tt.duplicated(CLAVE).sum()))
repetidos.sort_values(CLAVE)[CLAVE + ["Estado", col_dur_crudo]].rename(columns=utilidades.reparar_texto).head(6)

**Lectura.** No hay filas duplicadas. Un TT repetido es un **mismo ticket que agrupa varias caídas**
del elemento en fechas distintas (se ve en la tabla: mismo TT, distinto inicio). La clave
TT + elemento + inicio no se repite nunca. Cada fila es una caída distinta, y para medir la
recurrencia **hay que conservarlas todas**: borrar los TT repetidos borraría justo las reincidencias.

In [ ]:
registrar("2.3", "Conservar las filas con TT repetido",
          "un TT agrupa varias caídas del mismo elemento en fechas distintas; la clave TT + elemento + inicio es única",
          f"0 duplicados exactos; {repetidos['TT'].nunique()} TT en {len(repetidos)} filas conservadas")

### 2.4 Valores atípicos de la duración: medir antes de decidir

Se usa el criterio IQR: es atípico lo que queda fuera de `[Q1 − 1,5·IQR, Q3 + 1,5·IQR]`. Se mide
sobre las duraciones con formato válido, separando las fallas de transmisión del resto.

In [ ]:
def atipicos_iqr(serie, k=1.5):
    '''Límites IQR y cantidad de valores fuera de ellos. Ignora los vacíos.'''
    s = pd.Series(serie).dropna()
    if s.empty:
        raise utilidades.ProyectoError("no hay valores para calcular el IQR")
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - k * iqr, q3 + k * iqr
    fuera = (s < lim_inf) | (s > lim_sup)
    return {"n": len(s), "q1": q1, "q3": q3, "lim_sup": lim_sup, "atipicos": int(fuera.sum()),
            "pct_atipicos": round(fuera.mean() * 100, 1)}

texto_dur = crudo[col_dur_crudo].str.strip()
formato_ok = texto_dur.str.fullmatch(r"\d+:\d{2}:\d{2}")
minutos_explora = texto_dur[formato_ok].map(utilidades.duracion_a_minutos)
criticidad_explora = crudo.loc[formato_ok, "Criticidad"]
atipicos = pd.DataFrame({
    "transmisión": atipicos_iqr(minutos_explora[es_tx_crudo[formato_ok]]),
    "resto": atipicos_iqr(minutos_explora[~es_tx_crudo[formato_ok]]),
    "todas": atipicos_iqr(minutos_explora),
}).T.round(1)
atipicos

In [ ]:
lim_todas = atipicos.loc["todas", "lim_sup"]
largas = minutos_explora > lim_todas
print(f"Las atípicas por arriba son el {largas.mean():.1%} de las incidencias con duración")
print("Criticidad de las atípicas:", criticidad_explora[largas].value_counts().head(4).to_dict())
p3e = criticidad_explora == "P3E"
print(f"P3E sobre el límite: {int((largas & p3e).sum())} de {int(p3e.sum())}")

**Decisión.** Las duraciones atípicas **se conservan**. No son errores de registro: casi todas las
P3E (las que pasan mucho tiempo fuera de servicio) quedan sobre el límite, y en transmisión el
límite es de unas 79 horas. Son justo las caídas que bajan el uptime. Borrarlas o recortarlas cambiaría la respuesta a la pregunta. Para que no dominen el
escalamiento se trabaja con su logaritmo (sección 4.1).

In [ ]:
registrar("2.4", "Conservar las duraciones atípicas (IQR)",
          "son caídas largas reales, en buena parte P3E; son las que bajan el uptime",
          f"{int(atipicos.loc['todas', 'atipicos']):,} atípicas sobre {atipicos.loc['todas', 'lim_sup']:.0f} min ({atipicos.loc['todas', 'pct_atipicos']} %)".replace(",", "."))

### 2.5 Inconsistencias: duraciones inválidas y concordancia con las fechas

La duración debería ser igual al cierre menos el inicio. Se comprueba, y se revisan las 2
duraciones negativas que encontró la Fase 1.

In [ ]:
col_cierre_crudo = "Fecha Cierre de incidencia"
invalidas = texto_dur.ne("") & ~formato_ok
print("Duraciones con formato inválido:", int(invalidas.sum()))
print(crudo.loc[invalidas, ["Estado", "Tipo Incidencia NOC", col_cierre_crudo, col_dur_crudo]]
      .rename(columns=utilidades.reparar_texto).map(utilidades.reparar_texto).to_string())

inicio_explora = pd.to_datetime(crudo["Fecha de Inicio de incidencia"], format="%d-%m-%Y %H:%M")
cierre_explora = pd.to_datetime(crudo[col_cierre_crudo].where(~crudo[col_cierre_crudo].map(utilidades.es_vacio)),
                                format="%d-%m-%Y %H:%M")
diferencia = (minutos_explora - (cierre_explora - inicio_explora).dt.total_seconds() / 60).abs().dropna()
print(f"\nDiferencia entre la duración y (cierre − inicio): mediana {diferencia.median():.2f} min, "
      f"percentil 99 {diferencia.quantile(0.99):.2f} min, más de 5 min: {int((diferencia > 5).sum())} fila(s)")

**Lectura.** Las 2 duraciones negativas tienen como fecha de cierre `0000-00-00 00:00:00`, un valor
que el sistema escribe cuando no hay cierre real. La duración salió de restar esa fecha falsa. Las
dos son de energía y P4, fuera del foco. La duración y las fechas concuerdan con diferencias menores
a un minuto (el exporte redondea las fechas al minuto), así que la columna de duración es confiable.

## 3. Limpieza de datos

Desde aquí se trabaja sobre copias. `crudo` no se vuelve a tocar, y cada paso deja su tabla en una
variable nueva (`paso1`, `paso2`...) para poder comparar antes y después.

### 3.1 Nombres de columnas

El primer encabezado trae restos de HTML (`</div>` y una marca BOM) y varios tienen tildes rotas. Se
reparan y se pasan a `snake_case` sin tildes con `normalizar_columnas`, que además verifica que no
queden nombres repetidos. Luego se acortan los más largos.

In [ ]:
CORTOS = {
    "fecha_de_inicio_de_incidencia": "fecha_inicio", "fecha_cierre_de_incidencia": "fecha_cierre",
    "fecha_de_inicio_ticket": "fecha_inicio_ticket", "fecha_de_cierre_de_ticket": "fecha_cierre_ticket",
    "tiempo_de_duracion_de_la_incidencia": "duracion_txt", "tiempo_de_duracion_del_ticket": "duracion_ticket_txt",
    "network_element": "elemento_red", "q_network_element": "q_elementos", "tiempo_de_respuesta": "tiempo_respuesta_h",
}

def limpiar_nombres(df, cortos):
    '''Repara, normaliza y acorta los nombres de columna. Devuelve una copia.'''
    salida = df.copy()
    salida.columns = [cortos.get(n, n) for n in utilidades.normalizar_columnas(list(df.columns))]
    return salida

paso1 = limpiar_nombres(crudo, CORTOS)
print("Antes  :", list(crudo.columns[:3]))
print("Después:", list(paso1.columns[:3]))
assert paso1.columns.is_unique

### 3.2 Texto con tildes rotas

`reparar_texto` deshace la doble codificación (`EnergÃ­a` → `Energía`). Se aplica a todas las celdas
y se cuenta cuántas quedan con `Ã` o `Â` antes y después.

In [ ]:
def contar_rotas(df):
    return int(df.apply(lambda col: col.astype(str).str.contains("Ã|Â", regex=True)).sum().sum())

rotas_antes = contar_rotas(paso1)
paso2 = paso1.apply(lambda col: col.map(utilidades.reparar_texto))
rotas_despues = contar_rotas(paso2)
assert rotas_despues == 0, "Quedaron celdas con tildes rotas"
registrar("3.2", "Reparar la doble codificación del texto", "el exporte guardó el UTF-8 como Latin-1",
          f"{rotas_antes:,} celdas con tildes rotas → {rotas_despues}".replace(",", "."))
print(paso2["tipo_incidencia_noc"].value_counts().head(4).to_dict())

### 3.3 Vacíos escritos de varias formas y categorías mal escritas

Dos problemas distintos:

1. **"Sin dato" con varias formas** (`NA`, `N/A`, `na`, `no aplica`, `NA ` con espacio, la fecha
   `0000-00-00`): todas pasan a un único vacío (`NaN`).
2. **La misma categoría con distinta escritura** (`Pendiente` y `PENDIENTE`, `MOVISTAR` y `Movistar`):
   se agrupan sin distinguir mayúsculas ni espacios y se reemplazan por la forma más frecuente.

El texto libre (`titulo`, `detalle`) solo recibe el paso 1: sus valores no son categorías.

In [ ]:
TEXTO_LIBRE = ["titulo", "detalle"]

def unificar_vacios(df):
    '''Reemplaza todos los marcadores de "sin dato" por NaN y quita espacios extremos. Devuelve (df, conteo).'''
    salida = df.apply(lambda col: col.str.strip())
    marcadores = salida.apply(lambda col: col.map(utilidades.es_vacio))
    conteo = (marcadores & salida.ne("")).sum()          # los que estaban escritos (NA, N/A...), no los vacíos ""
    return salida.mask(marcadores), conteo[conteo > 0].sort_values(ascending=False)

def unificar_variantes(serie):
    '''Agrupa sin mayúsculas ni espacios dobles y usa la escritura más frecuente. Devuelve (serie, n_cambios).'''
    llave = serie.str.lower().str.replace(r"\s+", " ", regex=True)
    forma_frecuente = serie.groupby(llave).agg(lambda s: s.value_counts().index[0])
    nueva = llave.map(forma_frecuente).where(serie.notna())
    return nueva, int((nueva != serie)[serie.notna()].sum())

paso3, marcadores_escritos = unificar_vacios(paso2)
print("Marcadores escritos que pasaron a NaN, por columna:")
print(marcadores_escritos.head(8).to_string())

variantes = {}
CATEGORICAS = [c for c in paso3.columns if c not in TEXTO_LIBRE and paso3[c].nunique() <= 500]   # fechas e IDs no son categorías
for col in CATEGORICAS:
    paso3[col], cambios = unificar_variantes(paso3[col])
    if cambios:
        variantes[col] = cambios
print("\nValores reescritos a su forma más frecuente:", variantes)
registrar("3.3", "Unificar vacíos y variantes de escritura",
          "NA, N/A, na, no aplica y 0000-00-00 significan lo mismo; Pendiente y PENDIENTE también",
          f"{int(marcadores_escritos.sum()):,} marcadores → NaN; {sum(variantes.values())} valores reescritos en {len(variantes)} columnas".replace(",", "."))

### 3.4 Columnas fuera del alcance

Se quitan dos grupos, cada uno con su motivo:

- **Columnas casi vacías** (más de 95 % sin dato después de 3.3), salvo las que necesita la pregunta:
  `proveedor_tx_1` y `codigo_servicio_tx_1` están vacías en el total, pero **llenas en las fallas de
  transmisión**, que es donde se usan.
- **Columnas de otra clasificación o de otro foco:** las tres de GMI (una clasificación paralela a la
  del NOC) y `codigo_de_servicio` (código de suministro eléctrico, del foco de energía).

In [ ]:
CONSERVAR = ["proveedor_tx_1", "codigo_servicio_tx_1", "proveedor", "causa_raiz_noc"]
FUERA_DE_ALCANCE = ["tipo_incidencia_gmi", "incidenca_gmi", "causa_raiz_gmi", "codigo_de_servicio"]

def quitar_columnas(df, columnas):
    '''Quita columnas; falla con KeyError si alguna no existe (para no quitar nada por error de nombre).'''
    faltan = [c for c in columnas if c not in df.columns]
    if faltan:
        raise KeyError(f"no existen: {faltan}")
    return df.drop(columns=columnas)

es_tx3 = paso3["tipo_incidencia_noc"].isin(["Tx Propia", "Tx Proveedor"])
pct_vacio = paso3.isna().mean() * 100
casi_vacias = [c for c in pct_vacio[pct_vacio > 95].index if c not in CONSERVAR]
print("Vacío en el total vs en transmisión, columnas que se conservan:")
print(pd.DataFrame({"% vacío total": pct_vacio[CONSERVAR].round(1),
                    "% vacío en TX": (paso3.loc[es_tx3, CONSERVAR].isna().mean() * 100).round(1)}).to_string())
paso4 = quitar_columnas(paso3, casi_vacias + FUERA_DE_ALCANCE)
registrar("3.4", "Quitar columnas casi vacías y fuera de alcance",
          "más de 95 % sin dato, o pertenecen a la clasificación GMI o al foco de energía",
          f"{paso3.shape[1]} → {paso4.shape[1]} columnas ({len(casi_vacias)} casi vacías, {len(FUERA_DE_ALCANCE)} fuera de alcance)")
print("\nQuitadas:", casi_vacias + FUERA_DE_ALCANCE)

### 3.5 Tipos de dato

| Columna | De | A | Cómo |
|---|---|---|---|
| fechas | texto `dd-mm-aaaa hh:mm` | `datetime` | `pd.to_datetime` con formato explícito |
| `duracion_txt` | texto `h:mm:ss` | minutos (`float`) | `duracion_a_minutos`; las inválidas pasan a NaN con bandera |
| `q_elementos`, `tiempo_respuesta_h` | texto | entero | `astype(int)` |
| `gestionable`, `gestor_uptime` | `SI` / `NO` | 1 / 0 | mapa explícito; si aparece otro valor, falla |

In [ ]:
def a_minutos_seguro(serie):
    '''Convierte h:mm:ss a minutos. Las duraciones con formato inválido quedan NaN. Devuelve (minutos, invalidas).'''
    def convertir(valor):
        try:
            return utilidades.duracion_a_minutos(valor)
        except utilidades.ProyectoError:
            return "invalida"
    resultado = serie.map(convertir)
    invalidas = resultado.eq("invalida")
    return pd.to_numeric(resultado.mask(invalidas), errors="coerce"), invalidas

def a_binaria(serie, si="SI", no="NO"):
    '''SI/NO → 1/0. Lanza ProyectoError si aparece un valor distinto.'''
    otros = set(serie.dropna().str.upper()) - {si, no}
    if otros:
        raise utilidades.ProyectoError(f"valores no binarios en {serie.name}: {otros}")
    return serie.str.upper().map({si: 1, no: 0}).astype("Int64")

FORMATO_FECHA = "%d-%m-%Y %H:%M"
paso5 = paso4.copy()
for col in ["fecha_inicio", "fecha_cierre", "fecha_inicio_ticket", "fecha_cierre_ticket"]:
    paso5[col] = pd.to_datetime(paso5[col], format=FORMATO_FECHA)
paso5["duracion_min"], dur_invalida = a_minutos_seguro(paso5["duracion_txt"])
paso5["duracion_ticket_min"], _ = a_minutos_seguro(paso5["duracion_ticket_txt"])
paso5["cierre_invalido"] = dur_invalida.astype(int)
paso5 = paso5.drop(columns=["duracion_txt", "duracion_ticket_txt"])
for col in ["q_elementos", "tiempo_respuesta_h"]:
    paso5[col] = paso5[col].astype(int)
for col in ["gestionable", "gestor_uptime"]:
    paso5[col] = a_binaria(paso5[col])

registrar("3.5", "Convertir fechas, duración, conteos y banderas a su tipo",
          "el exporte entrega todo como texto",
          f"duración en minutos; {int(dur_invalida.sum())} duraciones inválidas → NaN con bandera cierre_invalido")
paso5.dtypes.value_counts()

### 3.6 Duraciones vacías: comparar los métodos antes de elegir

Se prueban cuatro alternativas sobre `duracion_min` y se mide qué le hace cada una a la media y a la
**desviación estándar**. Toda imputación con un valor central reduce la variabilidad; la pregunta es
cuánto, y si el método respeta el origen del vacío.

1. **Eliminar** las filas sin duración.
2. **Media** de todas las duraciones.
3. **Mediana** de todas las duraciones.
4. **Mediana por grupo** (tipo de incidencia NOC + criticidad): cada vacío recibe la mediana de las
   incidencias parecidas. Si un grupo no tiene ninguna duración, se usa la mediana general.

In [ ]:
def imputar_mediana_por_grupo(serie, grupos):
    '''Rellena los NaN con la mediana de su grupo; si el grupo no tiene datos, con la mediana general.'''
    mediana_grupo = serie.groupby(grupos).transform("median")
    return serie.fillna(mediana_grupo).fillna(serie.median())

def comparar_imputaciones(serie, metodos):
    '''Media, desviación estándar y % de cambio de la desviación para cada método.'''
    base_desv = serie.std()
    filas = []
    for nombre, funcion in metodos.items():
        resultado = funcion(serie)
        filas.append({"método": nombre, "n": int(resultado.notna().sum()), "media": resultado.mean(),
                      "desv_estandar": resultado.std(), "cambio_desv_pct": (resultado.std() / base_desv - 1) * 100})
    return pd.DataFrame(filas).set_index("método").round(1)

grupos = [paso5["tipo_incidencia_noc"], paso5["criticidad"]]
METODOS = {
    "sin tratar (referencia)": lambda s: s,
    "1. eliminar filas": lambda s: s.dropna(),
    "2. media": lambda s: s.fillna(s.mean()),
    "3. mediana": lambda s: s.fillna(s.median()),
    "4. mediana por grupo": lambda s: imputar_mediana_por_grupo(s, grupos),
}
vacias_dur = int(paso5["duracion_min"].isna().sum())
print(f"Duraciones vacías a tratar: {vacias_dur:,} ({vacias_dur / len(paso5):.1%})".replace(",", "."))
comparacion_imputacion = comparar_imputaciones(paso5["duracion_min"], METODOS)
comparacion_imputacion

**Decisión, en dos partes.**

- **En el conjunto limpio la duración vacía se deja vacía**, con la bandera `duracion_faltante`. El
  vacío corresponde a incidencias que aún no se normalizan (verificado en F1, sección 5.4) y casi
  todas son alarmas automáticas P4, fuera del foco. Inventarles una duración falsearía los minutos
  fuera de servicio, que es lo que mide la pregunta. Por eso la tabla de recurrencia usa solo
  duraciones reales.
- **En el conjunto para modelar se usa la mediana por grupo**, porque los modelos de las Fases 3 y 4
  no aceptan vacíos. De los cuatro métodos es el que **menos reduce la desviación estándar** sin
  borrar filas: respeta que una alarma automática P4 dura poco y una P3E dura días. La media y la
  mediana general achatan la distribución, y eliminar filas perdería una de cada cuatro
  incidencias. La bandera `duracion_faltante` queda en el modelo, para que el modelo sepa qué valores
  son imputados.

In [ ]:
paso6 = paso5.copy()
paso6["duracion_faltante"] = paso6["duracion_min"].isna().astype(int)
duracion_modelo = imputar_mediana_por_grupo(paso6["duracion_min"], [paso6["tipo_incidencia_noc"], paso6["criticidad"]])
fila = comparacion_imputacion.loc["4. mediana por grupo"]
registrar("3.6", "Duración vacía: se deja vacía en el limpio y mediana por grupo en el modelo, con bandera",
          "son incidencias no normalizadas (vacío no aleatorio); la mediana por grupo es la que menos reduce la desviación",
          f"{vacias_dur:,} vacías; cambio de la desviación con mediana por grupo: {fila['cambio_desv_pct']} %".replace(",", "."))

## 4. Transformación

### 4.1 Variables derivadas

| Variable | Qué es | Para qué sirve |
|---|---|---|
| `es_tx` | 1 si es Tx Propia o Tx Proveedor | separa el foco del resto |
| `tipo_enlace` | Microondas, Fibra propia (FOP), Fibra proveedor, Satelital u Otra TX | responde "qué tipo de enlace las origina" |
| `afecta_uptime` | 1 si la criticidad es P1, P2, P3 o P3E y no está cancelada | define la pérdida de servicio |
| `duracion_log` | logaritmo de (1 + minutos) | reduce el peso de las caídas de varios días al escalar |
| `hora_inicio`, `dia_semana` | momento en que empezó la caída | para las Fases 3 y 4 |
| `region_principal`, `n_regiones` | primera región y cantidad de regiones afectadas | `regiones` trae listas como "IX, VIII" |

Las incidencias **Canceladas** no cuentan para el uptime: el NOC las anula porque no correspondían a
una caída real.

In [ ]:
def derivar_variables(df):
    '''Agrega las variables derivadas de la tabla anterior. Devuelve una copia.'''
    salida = df.copy()
    salida["es_tx"] = salida["tipo_incidencia_noc"].isin(["Tx Propia", "Tx Proveedor"]).astype(int)
    salida["tipo_enlace"] = [utilidades.clasificar_enlace(t, i)
                             for t, i in zip(salida["tipo_incidencia_noc"], salida["incidencia_noc"])]
    salida["afecta_uptime"] = (salida["criticidad"].isin(utilidades.CRITICIDAD_UPTIME)
                               & salida["estado"].ne("Cancelada")).astype(int)
    salida["duracion_log"] = np.log1p(salida["duracion_min"])
    salida["hora_inicio"] = salida["fecha_inicio"].dt.hour
    salida["dia_semana"] = salida["fecha_inicio"].dt.day_of_week          # 0 = lunes
    regiones = salida["regiones"].fillna("").str.split(",")
    salida["region_principal"] = regiones.str[0].str.strip().replace("", np.nan)
    salida["n_regiones"] = regiones.map(lambda r: len([x for x in r if x.strip()]))
    return salida

paso7 = derivar_variables(paso6)
print("Fallas de transmisión:", int(paso7["es_tx"].sum()), "· con pérdida de servicio:",
      int((paso7["es_tx"] & paso7["afecta_uptime"]).sum()))
pd.crosstab(paso7["tipo_enlace"], paso7["afecta_uptime"].map({1: "con pérdida", 0: "sin pérdida"}), margins=True, margins_name="Total")

In [ ]:
registrar("4.1", "Derivar es_tx, tipo_enlace, afecta_uptime, duracion_log y variables de tiempo y región",
          "la pregunta pide el tipo de enlace y la pérdida de servicio; ninguna viene como columna",
          f"{int(paso7['es_tx'].sum())} fallas TX clasificadas en {paso7['tipo_enlace'].nunique()} tipos de enlace; "
          f"{int(paso7['afecta_uptime'].sum()):,} incidencias afectan el uptime".replace(",", "."))

### 4.2 Variables ordinales: el orden se declara, no se deja al alfabeto

Si se codifica la criticidad en orden alfabético, `P2 Redundancia` queda entre `P2` y `P2R`, y `P3E`
antes de `P4` solo por casualidad. El orden se declara explícitamente, **de menor a mayor gravedad**,
y `codificar_ordinal` falla si aparece un valor que no está en la lista, para que ningún valor
nuevo pase en silencio.

In [ ]:
ORDENES = {
    "criticidad": ["P4", "P3", "P3E", "P2R Degradación", "P2 Redundancia", "P2", "P1"],
    "impacto": ["Sin Impacto", "Localizado", "Regional", "Amplio"],
}

def codificar_ordinal(serie, orden):
    '''Convierte una categoría ordinal en su posición (0 = menor) según el orden declarado.'''
    desconocidos = set(serie.dropna()) - set(orden)
    if desconocidos:
        raise utilidades.ProyectoError(f"{serie.name}: valores fuera del orden declarado {desconocidos}")
    return serie.map({valor: i for i, valor in enumerate(orden)}).astype("Int64")

paso8 = paso7.copy()
for variable, orden in ORDENES.items():
    paso8[f"{variable}_ord"] = codificar_ordinal(paso8[variable], orden)
print(paso8.groupby("criticidad")["criticidad_ord"].first().sort_values().to_string())
registrar("4.2", "Codificar criticidad e impacto con orden declarado",
          "el orden alfabético no respeta la gravedad",
          f"criticidad 0 (P4) a {len(ORDENES['criticidad']) - 1} (P1); impacto 0 (Sin Impacto) a 3 (Amplio)")

### 4.3 Variables nominales: one-hot con el patrón `fit` / `transform`

`OneHotEncoder` crea una columna 0/1 por categoría. Las categorías con menos de 1 % de las filas se
agrupan en una sola columna `infrequent`, para no crear decenas de columnas casi vacías. La
excepción es `tipo_enlace`: microondas y FOP son menos del 1 % del total, pero son el foco de la
pregunta, así que cada tipo de enlace conserva su columna. El
codificador se **ajusta** (`fit`) y luego se **aplica** (`transform`): en la Fase 3 se ajustará solo
con el conjunto de entrenamiento.

In [ ]:
NOMINALES = ["tipo_incidencia_noc", "estado", "region_principal", "tipo_enlace"]
FREC_MINIMA = 0.01

def codificar_nominales(df, columnas, frec_minima):
    '''One-hot de las columnas nominales. Los vacíos pasan a la categoría "Sin dato". Devuelve (tabla, codificador).'''
    entrada = df[columnas].fillna("Sin dato")
    codificador = OneHotEncoder(min_frequency=frec_minima, handle_unknown="infrequent_if_exist", sparse_output=False)
    codificador.fit(entrada)
    nombres = [utilidades.normalizar_nombre(n) for n in codificador.get_feature_names_out(columnas)]
    return pd.DataFrame(codificador.transform(entrada).astype(int), columns=nombres, index=df.index), codificador

# tipo_enlace no se agrupa: microondas y FOP son menos del 1 % del total, pero son el foco de la pregunta
onehot_general, _ = codificar_nominales(paso8, ["tipo_incidencia_noc", "estado", "region_principal"], FREC_MINIMA)
onehot_enlace, _ = codificar_nominales(paso8, ["tipo_enlace"], None)
onehot = pd.concat([onehot_general, onehot_enlace], axis=1)
print("Columnas one-hot por variable:",
      {c: sum(n.startswith(c + "_") for n in onehot.columns) for c in NOMINALES})
registrar("4.3", "One-hot de tipo de incidencia, estado, región y tipo de enlace (frecuencia mínima 1 %)",
          "son nominales: no tienen orden",
          f"{len(NOMINALES)} variables → {onehot.shape[1]} columnas 0/1; tipo_enlace sin agrupar")
onehot.head(3)

### 4.4 Recurrencia por elemento de red (la respuesta a la pregunta) y la función recursiva

Para cada elemento de red se cuentan sus fallas de transmisión **con pérdida de servicio** y sus
minutos fuera de servicio.

Las incidencias **Canceladas** no se cuentan (4 fallas TX con criticidad de pérdida de servicio
fueron canceladas): por eso aquí hay 260 fallas y no las 264 que contó la Fase 1.

**Por qué hace falta una función recursiva.** Si un elemento tiene dos incidencias abiertas al mismo
tiempo (por ejemplo, una caída de microondas y otra de fibra que se traslapan), sumar sus duraciones
cuenta dos veces el mismo tiempo. `utilidades.fusionar_intervalos` une los intervalos que se
traslapan, de forma **recursiva**: el caso base es una lista de 0 o 1 intervalo; en el paso recursivo
fusiona el resto de la lista y decide si el primer intervalo se une al siguiente o queda aparte.
Así se obtiene el tiempo **real** fuera de servicio.

In [ ]:
def minutos_reales(inicios, cierres):
    '''Minutos fuera de servicio sin contar dos veces los traslapes (usa la fusión recursiva).'''
    intervalos = list(zip(inicios, cierres))
    return sum((fin - ini).total_seconds() / 60 for ini, fin in utilidades.fusionar_intervalos(intervalos))

def tabla_recurrencia(df):
    '''Una fila por elemento de red: fallas TX con pérdida de servicio, minutos sumados y minutos reales.'''
    foco = df[(df["es_tx"] == 1) & (df["afecta_uptime"] == 1)]
    con_cierre = foco.dropna(subset=["fecha_cierre"])
    tabla = foco.groupby("elemento_red").agg(
        fallas=("tt", "size"),
        fallas_p3e=("criticidad", lambda s: int((s == "P3E").sum())),
        abiertas=("fecha_cierre", lambda s: int(s.isna().sum())),
        enlace_principal=("tipo_enlace", lambda s: s.value_counts().index[0]),
        proveedor_tx=("proveedor_tx_1", lambda s: s.value_counts().index[0] if s.notna().any() else "Sin dato"),
        region=("region_principal", lambda s: s.value_counts().index[0] if s.notna().any() else "Sin dato"),
    )
    duracion_fechas = (con_cierre["fecha_cierre"] - con_cierre["fecha_inicio"]).dt.total_seconds() / 60
    tabla["minutos_sumados"] = duracion_fechas.groupby(con_cierre["elemento_red"]).sum()
    tabla["minutos_reales"] = pd.Series({elemento: minutos_reales(g["fecha_inicio"], g["fecha_cierre"])
                                         for elemento, g in con_cierre.groupby("elemento_red")})
    tabla[["minutos_sumados", "minutos_reales"]] = tabla[["minutos_sumados", "minutos_reales"]].fillna(0).round(0)
    tabla["horas_fuera"] = (tabla["minutos_reales"] / 60).round(1)
    return tabla.sort_values(["fallas", "minutos_reales"], ascending=False).reset_index()

recurrencia = tabla_recurrencia(paso8)
traslape = recurrencia["minutos_sumados"].sum() - recurrencia["minutos_reales"].sum()
print(f"Elementos con pérdida de servicio por transmisión: {len(recurrencia)} · "
      f"con 2 o más: {int((recurrencia['fallas'] >= 2).sum())} · con 3 o más: {int((recurrencia['fallas'] >= 3).sum())}")
print(f"Minutos contados dos veces por traslape (y que la función recursiva descuenta): {traslape:,.0f}".replace(",", "."))
registrar("4.4", "Tabla de recurrencia por elemento con minutos reales (fusión recursiva de intervalos)",
          "sumar duraciones traslapadas cuenta dos veces el mismo tiempo fuera de servicio",
          f"{len(recurrencia)} elementos; {int((recurrencia['fallas'] >= 2).sum())} con 2 o más fallas; {traslape:,.0f} min de traslape descontados".replace(",", "."))
recurrencia.head(15)

In [ ]:
def grafico_recurrencia(tabla, ruta, n=15):
    '''Barras de los n elementos con más fallas, coloreadas por tipo de enlace, con las horas fuera de servicio.'''
    top = tabla.head(n).iloc[::-1]
    colores = {"Microondas": "#3B6FD8", "Fibra propia (FOP)": "#2E9E6A", "Fibra proveedor": "#E08A2E",
               "Satelital": "#8E5BC9", "Otra TX": "#7F8C8D"}
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(top["elemento_red"], top["fallas"], color=[colores[e] for e in top["enlace_principal"]])
    for y, (f, h) in enumerate(zip(top["fallas"], top["horas_fuera"])):
        ax.text(f + 0.1, y, f"{h:.0f} h", va="center", fontsize=8)
    ax.set_xlabel("Fallas de transmisión con pérdida de servicio (octubre 2026)")
    ax.set_title(f"Los {n} elementos de red que más fallan · etiqueta: horas fuera de servicio")
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in colores.values()],
              labels=list(colores), fontsize=8, loc="lower right")
    fig.tight_layout()
    fig.savefig(ruta, dpi=150)
    plt.show()

grafico_recurrencia(recurrencia, DIR_FIG / "f2_recurrencia_top15.png")

In [ ]:
# Resumen por tipo de enlace y proveedor: a qué área escalar
foco = paso8[(paso8["es_tx"] == 1) & (paso8["afecta_uptime"] == 1)]
(foco.groupby(["tipo_enlace", foco["proveedor_tx_1"].fillna("Sin dato")])
     .agg(fallas=("tt", "size"), elementos=("elemento_red", "nunique"), mediana_min=("duracion_min", "median"))
     .round(0).sort_values("fallas", ascending=False).head(10))

## 5. Escalamiento

Las variables numéricas tienen escalas muy distintas: la duración va de 0 a más de 50 mil minutos,
la cantidad de elementos de 1 a 72 y el tiempo de respuesta de 0 a 48 horas. Se comparan tres
escaladores antes de elegir:

- **StandardScaler:** resta la media y divide por la desviación estándar.
- **MinMaxScaler:** lleva todo al rango 0 a 1.
- **RobustScaler:** resta la mediana y divide por el IQR; no se deja arrastrar por los extremos.

In [ ]:
NUMERICAS = ["duracion_log", "q_elementos", "tiempo_respuesta_h"]
ESCALADORES = {"StandardScaler": StandardScaler(), "MinMaxScaler": MinMaxScaler(), "RobustScaler": RobustScaler()}

def comparar_escaladores(df, columnas, escaladores):
    '''Ajusta cada escalador y resume media, desviación, mínimo, mediana y máximo del resultado.'''
    filas = []
    for nombre, escalador in escaladores.items():
        for col in columnas:
            z = escalador.fit_transform(df[[col]].dropna()).ravel()
            filas.append({"escalador": nombre, "variable": col, "media": z.mean(), "desv": z.std(),
                          "min": z.min(), "mediana": np.median(z), "max": z.max()})
    return pd.DataFrame(filas).set_index(["escalador", "variable"]).round(2)

paso8["duracion_log_modelo"] = np.log1p(duracion_modelo)
print(paso8[NUMERICAS].describe().round(2).to_string(), "\n")
comparar_escaladores(paso8, NUMERICAS, ESCALADORES)

**Decisión.**

- `duracion_log`: después del logaritmo la distribución es mucho más pareja, y **StandardScaler** la
  deja centrada en 0 con desviación 1 sin valores extremos que dominen.
- `q_elementos`: casi todas las incidencias afectan a 1 elemento (mediana 1, IQR 0) y unas pocas a
  decenas. Con un IQR de 0, RobustScaler no escala nada (deja valores de 0 a 71), y StandardScaler
  lleva esas pocas hasta 43 desviaciones. **MinMaxScaler** la deja entre 0 y 1 y conserva el orden.
- `tiempo_respuesta_h`: toma solo cinco valores (0, 2, 4, 24 y 48 horas). **MinMaxScaler** los lleva a
  0–1 y conserva la distancia entre ellos.

El ajuste se hace sobre todo el conjunto porque esta fase no separa entrenamiento y prueba. **En la
Fase 3 los escaladores se ajustarán solo con el conjunto de entrenamiento**, para no filtrar información.

In [ ]:
def escalar(df, plan):
    '''Aplica {columna_origen: (columna_destino, escalador)}. Devuelve (df, escaladores ajustados).'''
    salida = df.copy()
    ajustados = {}
    for origen, (destino, escalador) in plan.items():
        salida[destino] = escalador.fit_transform(salida[[origen]]).ravel()
        ajustados[destino] = escalador
    return salida, ajustados

PLAN_ESCALA = {"duracion_log_modelo": ("duracion_log_esc", StandardScaler()),
               "q_elementos": ("q_elementos_esc", MinMaxScaler()),
               "tiempo_respuesta_h": ("tiempo_respuesta_esc", MinMaxScaler())}
paso9, escaladores = escalar(paso8, PLAN_ESCALA)
registrar("5", "StandardScaler en duracion_log; MinMaxScaler en q_elementos y tiempo_respuesta",
          "duracion_log es casi simétrica; q_elementos tiene IQR 0; tiempo_respuesta toma 5 valores",
          f"media de duracion_log_esc = {paso9['duracion_log_esc'].mean():.2f}, desv = {paso9['duracion_log_esc'].std(ddof=0):.2f}")
paso9[[d for d, _ in PLAN_ESCALA.values()]].describe().round(2)

### 5.1 Armado de los conjuntos de salida

- **`incidencias_limpio.csv`**: una fila por incidencia, legible, con las duraciones reales (las vacías
  quedan vacías) y las variables derivadas y ordinales. Es el que se lee para explorar en las Fases 3 y 4.
- **`incidencias_modelo.csv`**: solo columnas numéricas y sin vacíos: ordinales, banderas, one-hot y
  numéricas escaladas. Es la matriz para modelar.
- **`recurrencia_tx_por_elemento.csv`**: la tabla de la sección 4.4.

In [ ]:
ID = "tt"
COLUMNAS_LIMPIO = [
    "tt", "elemento_red", "fecha_inicio", "fecha_cierre", "estado", "tipo_incidencia_noc", "incidencia_noc",
    "causa_raiz_noc", "criticidad", "criticidad_ord", "impacto", "impacto_ord", "afecta_uptime", "es_tx",
    "tipo_enlace", "proveedor_tx_1", "codigo_servicio_tx_1", "proveedor", "duracion_min", "duracion_faltante",
    "cierre_invalido", "duracion_log", "q_elementos", "tiempo_respuesta_h", "hora_inicio", "dia_semana",
    "region_principal", "n_regiones", "comuna", "gestionable", "gestor_uptime", "titulo", "detalle",
]
COLUMNAS_MODELO = (["criticidad_ord", "impacto_ord", "afecta_uptime", "es_tx", "duracion_faltante", "cierre_invalido",
                    "gestionable", "gestor_uptime", "hora_inicio", "dia_semana", "n_regiones"]
                   + [d for d, _ in PLAN_ESCALA.values()])

limpio = paso9[COLUMNAS_LIMPIO].copy()
modelo = pd.concat([paso9[[ID] + COLUMNAS_MODELO], onehot], axis=1)
modelo[COLUMNAS_MODELO] = modelo[COLUMNAS_MODELO].astype(float)
print(f"limpio: {limpio.shape} · modelo: {modelo.shape} · recurrencia: {recurrencia.shape}")

## 6. Validación técnica y verificación del código

### 6.1 Integridad del conjunto final

Comprobaciones con `assert` sobre lo que el pipeline promete. Si una falla, el notebook se detiene en
el punto exacto y dice cuál.

In [ ]:
def columnas_texto(df):
    '''Columnas que guardan texto (ni números ni fechas).'''
    return [c for c in df.columns if not (pd.api.types.is_numeric_dtype(df[c]) or pd.api.types.is_datetime64_any_dtype(df[c]))]

def validar_conjunto(limpio, modelo, recurrencia, filas_esperadas):
    '''Reglas de integridad del resultado. Devuelve una tabla con cada regla y su resultado.'''
    grupos_onehot = {n: [c for c in modelo.columns if c.startswith(n + "_")] for n in NOMINALES}
    cerradas = limpio.dropna(subset=["fecha_cierre", "duracion_min"])
    reglas = {
        "las 21.542 filas siguen en limpio y en modelo": len(limpio) == len(modelo) == filas_esperadas,
        "clave TT + elemento + inicio única": not limpio.duplicated(["tt", "elemento_red", "fecha_inicio"]).any(),
        "sin celdas con tildes rotas": contar_rotas(limpio.astype(str)) == 0,
        "sin marcadores NA, N/A o no aplica escritos": not any(
            limpio[c].dropna().map(utilidades.es_vacio).any() for c in columnas_texto(limpio)),
        "modelo sin vacíos y todo numérico": (not modelo.drop(columns=[ID]).isna().any().any())
            and all(pd.api.types.is_numeric_dtype(modelo[c]) for c in modelo.columns if c != ID),
        "duraciones no negativas": (limpio["duracion_min"].dropna() >= 0).all(),
        "duración = cierre − inicio (± 2 min)": ((cerradas["duracion_min"]
            - (cerradas["fecha_cierre"] - cerradas["fecha_inicio"]).dt.total_seconds() / 60).abs() <= 2).mean() >= 0.999,
        "duracion_faltante coincide con duración vacía": (limpio["duracion_faltante"] == limpio["duracion_min"].isna()).all(),
        "ordinales dentro de su rango": all(limpio[f"{v}_ord"].between(0, len(o) - 1).all() for v, o in ORDENES.items()),
        "cada grupo one-hot suma 1 por fila": all((modelo[c].sum(axis=1) == 1).all() for c in grupos_onehot.values()),
        "tipo_enlace definido en todas las fallas TX y solo en ellas": limpio["tipo_enlace"].notna().eq(limpio["es_tx"] == 1).all(),
        "afecta_uptime solo en P1, P2, P3 y P3E": set(limpio.loc[limpio["afecta_uptime"] == 1, "criticidad"]) <= set(utilidades.CRITICIDAD_UPTIME),
        "minutos reales ≤ minutos sumados": (recurrencia["minutos_reales"] <= recurrencia["minutos_sumados"] + 0.5).all(),
        "recurrencia suma las fallas TX con pérdida de servicio": recurrencia["fallas"].sum() == int(((limpio["es_tx"] == 1) & (limpio["afecta_uptime"] == 1)).sum()),
        "data/raw sin cambios (SHA-256)": utilidades.huella_sha256(conjunto.ruta) == HUELLA_RAW,
    }
    return pd.DataFrame({"regla": list(reglas), "resultado": ["OK" if v else "FALLA" for v in reglas.values()]})

validacion = validar_conjunto(limpio, modelo, recurrencia, FILAS_ORIGINALES)
print(validacion.to_string(index=False))
assert (validacion["resultado"] == "OK").all(), "Hay reglas de integridad que no se cumplen"
print(f"\n[OK] {len(validacion)} de {len(validacion)} reglas de integridad")

### 6.2 Pruebas de las funciones: caso normal, caso límite y excepción

Cada función del pipeline se prueba con datos pequeños hechos a mano, donde el resultado correcto
se conoce de antemano. Si alguna falla, el `assert` detiene el notebook.

In [ ]:
from datetime import datetime as dt

# --- fusionar_intervalos (la función recursiva) ---
assert utilidades.fusionar_intervalos([(1, 3), (2, 5), (7, 8)]) == [(1, 5), (7, 8)]                      # normal
assert utilidades.fusionar_intervalos([(0, 10), (1, 2), (3, 4)]) == [(0, 10)]                            # uno contiene a otros
assert utilidades.fusionar_intervalos([]) == [] and utilidades.fusionar_intervalos([(4, 4)]) == [(4, 4)]  # límite: 0 y 1 intervalo
assert utilidades.fusionar_intervalos([(1, 2), (2, 3)]) == [(1, 3)]                                      # límite: se tocan
try:                                                                                                     # excepción
    utilidades.fusionar_intervalos([(5, 1)])
    raise AssertionError("debió fallar")
except utilidades.ProyectoError as e:
    print("[OK] fusionar_intervalos · normal, contenido, 0 y 1 intervalo, se tocan; excepción:", e)

# --- minutos_reales: dos caídas que se traslapan una hora ---
assert minutos_reales([dt(2026, 10, 1, 8), dt(2026, 10, 1, 9)], [dt(2026, 10, 1, 10), dt(2026, 10, 1, 11)]) == 180
print("[OK] minutos_reales · 2 caídas de 2 h con 1 h de traslape = 3 h, no 4 h")

# --- clasificar_enlace ---
assert utilidades.clasificar_enlace("Tx Propia", "MW Pérdida de enlace") == "Microondas"
assert utilidades.clasificar_enlace("Tx Propia", "FOP Corte de Fibra") == "Fibra propia (FOP)"
assert utilidades.clasificar_enlace("Tx Proveedor", "Pérdida de Redundancia") == "Fibra proveedor"
assert utilidades.clasificar_enlace("Tx Proveedor", "Falla Modem Satelital (Subtel)") == "Satelital"
assert utilidades.clasificar_enlace("Energía", "Corte de energía") is None                               # límite: no es TX
try:
    utilidades.clasificar_enlace("Tx Propia", "")
    raise AssertionError("debió fallar")
except utilidades.ProyectoError as e:
    print("[OK] clasificar_enlace · 4 tipos, no TX → None; excepción:", e)

# --- codificar_ordinal ---
assert codificar_ordinal(pd.Series(["P1", "P4", None], name="c"), ORDENES["criticidad"]).tolist()[:2] == [6, 0]
assert codificar_ordinal(pd.Series([], dtype=object, name="vacia"), ["A"]).empty                          # límite: vacía
try:
    codificar_ordinal(pd.Series(["P9"], name="criticidad"), ORDENES["criticidad"])
    raise AssertionError("debió fallar")
except utilidades.ProyectoError as e:
    print("[OK] codificar_ordinal · normal, vacía; excepción:", e)

# --- imputar_mediana_por_grupo ---
v = pd.Series([1.0, np.nan, 3.0, np.nan, 10.0])
g = pd.Series(["a", "a", "a", "b", "c"])
assert imputar_mediana_por_grupo(v, g).tolist() == [1.0, 2.0, 3.0, 3.0, 10.0]   # grupo 'b' sin datos → mediana general
assert v.isna().sum() == 2, "No debe modificar la entrada"
print("[OK] imputar_mediana_por_grupo · normal, grupo sin datos y entrada intacta")

# --- a_minutos_seguro y a_binaria ---
m, inv = a_minutos_seguro(pd.Series(["1:30:00", "", "-5:-1:00"]))
assert m.iloc[0] == 90 and np.isnan(m.iloc[1]) and inv.tolist() == [False, False, True]
assert a_binaria(pd.Series(["SI", "no"], name="x")).tolist() == [1, 0]
try:
    a_binaria(pd.Series(["TAL VEZ"], name="gestionable"))
    raise AssertionError("debió fallar")
except utilidades.ProyectoError as e:
    print("[OK] a_minutos_seguro y a_binaria · normal, vacía, inválida; excepción:", e)

# --- unificar_variantes, es_vacio, atipicos_iqr, quitar_columnas ---
u, n = unificar_variantes(pd.Series(["Pendiente", "PENDIENTE", "Pendiente", None]))
assert u.tolist()[:3] == ["Pendiente"] * 3 and n == 1
assert all(utilidades.es_vacio(x) for x in ["NA ", "N/A", "no aplica", "0000-00-00 00:00:00", None]) and not utilidades.es_vacio("WOM")
assert atipicos_iqr(pd.Series([1, 2, 3, 4, 100]))["atipicos"] == 1
try:
    atipicos_iqr(pd.Series([np.nan]))
    raise AssertionError("debió fallar")
except utilidades.ProyectoError:
    pass
try:
    quitar_columnas(pd.DataFrame({"a": [1]}), ["b"])
    raise AssertionError("debió fallar")
except KeyError as e:
    print("[OK] unificar_variantes, es_vacio, atipicos_iqr y quitar_columnas · excepción:", e)

### 6.3 Conteos antes y después

La trazabilidad de cada etapa: ninguna fila se pierde en la limpieza y las columnas cambian solo
donde se decidió.

In [ ]:
trazas = pd.DataFrame([
    ("original", FILAS_ORIGINALES, COLUMNAS_ORIGINALES),
    ("3.1 nombres normalizados", *paso1.shape),
    ("3.3 vacíos y variantes unificados", *paso3.shape),
    ("3.4 columnas fuera de alcance quitadas", *paso4.shape),
    ("3.5 tipos convertidos", *paso5.shape),
    ("4 variables derivadas y ordinales", *paso8.shape),
    ("salida: incidencias_limpio", *limpio.shape),
    ("salida: incidencias_modelo", *modelo.shape),
    ("salida: recurrencia_tx_por_elemento", *recurrencia.shape),
], columns=["etapa", "filas", "columnas"]).set_index("etapa")
trazas

## 7. Persistencia y trazabilidad

Se guardan los tres conjuntos y se **releen** para comprobar que lo guardado es igual a lo que está en
memoria. También se guardan el diccionario del conjunto para modelar, los metadatos (con la huella
de cada archivo y el commit con que se generó) y la bitácora de decisiones.

In [ ]:
def git(*argumentos):
    '''Ejecuta un comando de Git en la raíz del proyecto y devuelve su salida.'''
    r = subprocess.run(["git", *argumentos], cwd=RAIZ, capture_output=True, text=True, encoding="utf-8")
    return (r.stdout or r.stderr).strip()

SALIDAS = {DIR_PROC / "incidencias_limpio.csv": limpio, DIR_PROC / "incidencias_modelo.csv": modelo,
           DIR_PROC / "recurrencia_tx_por_elemento.csv": recurrencia}
for ruta, tabla in SALIDAS.items():
    tabla.to_csv(ruta, index=False, encoding="utf-8")
    releido = pd.read_csv(ruta, low_memory=False)
    assert releido.shape == tabla.shape and list(releido.columns) == list(tabla.columns), f"{ruta.name} no se releyó igual"
releido_modelo = pd.read_csv(DIR_PROC / "incidencias_modelo.csv")
assert np.allclose(releido_modelo.drop(columns=[ID]).values, modelo.drop(columns=[ID]).values)
print("[OK] los tres archivos se releyeron con la misma forma, columnas y valores")

diccionario_modelo = pd.DataFrame({
    "variable": modelo.columns,
    "grupo": ["identificador" if c == ID else "one-hot" if c in onehot.columns else
              "escalada" if c.endswith("_esc") else "ordinal" if c.endswith("_ord") else "bandera o derivada"
              for c in modelo.columns],
    "dtype": modelo.dtypes.astype(str).values,
})
diccionario_modelo.to_csv(DIR_DOCS / "diccionario_modelo.csv", index=False, encoding="utf-8")

meta_f2 = {
    "fase": "F2", "generado": datetime.now().isoformat(timespec="seconds"), "commit_base": git("rev-parse", "--short", "HEAD"),
    "entrada": {"archivo": meta_f1["dataset"]["archivo"], "sha256": HUELLA_RAW, "filas": FILAS_ORIGINALES},
    "salidas": {r.name: {"filas": int(t.shape[0]), "columnas": int(t.shape[1]), "sha256": utilidades.huella_sha256(r)}
                for r, t in SALIDAS.items()},
    "parametros": {"semilla": SEMILLA, "frecuencia_minima_onehot": FREC_MINIMA, "criticidad_uptime": list(utilidades.CRITICIDAD_UPTIME),
                   "orden_criticidad": ORDENES["criticidad"], "orden_impacto": ORDENES["impacto"],
                   "escaladores": {d: type(e).__name__ for d, e in escaladores.items()}},
    "resultados": {"fallas_tx": int(limpio["es_tx"].sum()),
                   "fallas_tx_con_perdida": int(((limpio["es_tx"] == 1) & (limpio["afecta_uptime"] == 1)).sum()),
                   "elementos_con_2_o_mas": int((recurrencia["fallas"] >= 2).sum()),
                   "minutos_traslape_descontados": float(traslape)},
    "validacion": dict(zip(validacion["regla"], validacion["resultado"])),
}
(DIR_DOCS / "metadatos_fase2.json").write_text(json.dumps(meta_f2, ensure_ascii=False, indent=2), encoding="utf-8")

lineas = ["# Bitácora de decisiones · Fase 2", "",
          "Generada por `F2/F2_Preprocesamiento.ipynb`. Una fila por decisión, con su motivo y su impacto medido.", "",
          "| Paso | Decisión | Motivo | Impacto medido |", "|---|---|---|---|"]
lineas += [f"| {b['paso']} | {b['decision']} | {b['motivo']} | {b['impacto']} |" for b in BITACORA]
(DIR_DOCS / "bitacora_decisiones.md").write_text("\n".join(lineas) + "\n", encoding="utf-8")

for ruta in [*SALIDAS, DIR_DOCS / "diccionario_modelo.csv", DIR_DOCS / "metadatos_fase2.json", DIR_DOCS / "bitacora_decisiones.md"]:
    print(f"{ruta.relative_to(RAIZ).as_posix():<45} {ruta.stat().st_size / 1e6:6.2f} MB")
pd.DataFrame(BITACORA)

## 8. Conclusiones y reflexión técnica

**Hallazgos.**

- **La pregunta tiene respuesta con datos limpios.** La tabla de recurrencia (sección 4.4) ordena los
  elementos de red por sus fallas de transmisión con pérdida de servicio y sus horas reales fuera de
  servicio, con el tipo de enlace y el proveedor. Es la lista que se lleva al área de transmisión o
  al proveedor para evaluar un cambio de enlace.
- **Contar solo la pérdida de servicio cambia la prioridad.** Las fallas sin pérdida de servicio
  (pérdidas de redundancia, P4) no bajan el uptime. Si se cuentan, suben a la lista elementos que
  fallan seguido pero no dejan el sitio caído.
- **El tiempo fuera de servicio se puede contar dos veces.** Cuando dos incidencias del mismo
  elemento se traslapan, sumar sus duraciones infla el total. En octubre el efecto fue pequeño
  (30 minutos), pero la función recursiva `fusionar_intervalos` garantiza que no ocurra en otros
  meses, cuando se repita el análisis.
- **Los TT repetidos no eran duplicados**, sino reincidencias del mismo elemento agrupadas en un
  ticket. Borrarlos habría borrado justo la recurrencia que se quiere medir.

**Dificultades y cómo se resolvieron.** Las 2 duraciones negativas salían de una fecha de cierre
falsa (`0000-00-00`); se detectó cruzando la duración con las fechas. Varias columnas parecían
vacías en el total y se habrían quitado, pero están llenas en las fallas de transmisión; se midió el
vacío dentro del foco antes de decidir.

**Decisiones abiertas para la Fase 3.**

- Ajustar el one-hot y los escaladores solo con el conjunto de entrenamiento.
- Analizar la recurrencia por semana, para ver si las fallas de un elemento se concentran en pocos días.
- Las incidencias aún abiertas tienen tiempo fuera de servicio desconocido; si se necesitan, medir
  el tiempo transcurrido hasta la fecha del exporte como cota mínima.

## 9. Verificación antes de entregar

- [ ] Corre completo con *Kernel → Restart Kernel and Run All Cells* y la numeración es continua.
- [ ] Cada etapa abre con una celda narrativa: obtener → explorar → limpiar → transformar → escalar → validar → guardar.
- [ ] Cada decisión de limpieza se tomó después de medir y quedó en la bitácora con su cifra.
- [ ] `data/raw` no cambió (huella SHA-256 verificada al inicio y al final).
- [ ] `data/processed` contiene los tres productos y se releyeron sin diferencias.
- [ ] Pruebas con casos normales, límite y excepciones, con su salida conservada.
- [ ] Las salidas quedaron guardadas (Ctrl+S) **antes** del commit.